In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, recall_score,precision_score, make_scorer, accuracy_score
from sklearn.preprocessing import StandardScaler
#from xgboost import XGBClassifier

In [5]:
data=pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
data.info(verbose=True)
data.shape
data.duplicated()
data[data.duplicated(['Diabetes_binary','HighChol'] )]
data.drop_duplicates(inplace=True)
data.duplicated().any()
print(data.shape)
data.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_binary       253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  MentHlth   

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
5,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,2.0,0.0,1.0,10.0,6.0,8.0
6,0.0,1.0,0.0,1.0,30.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,3.0,0.0,14.0,0.0,0.0,9.0,6.0,7.0
7,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,0.0,0.0,1.0,0.0,11.0,4.0,4.0
8,1.0,1.0,1.0,1.0,30.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,5.0,30.0,30.0,1.0,0.0,9.0,5.0,1.0
9,0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,2.0,0.0,0.0,0.0,1.0,8.0,4.0,3.0


In [6]:
data.describe()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.00000,229474.000000,229474.000000,229474.000000,229474.000000,...,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000
mean,0.152945,0.454343,0.441640,0.959481,28.687507,0.46580,0.044816,0.103336,0.733042,0.612675,...,0.946011,0.092921,2.601820,3.509866,4.681219,0.185751,0.439087,8.085068,4.979741,5.888615
std,0.359936,0.497912,0.496584,0.197173,6.789204,0.49883,0.206899,0.304398,0.442371,0.487140,...,0.225996,0.290323,1.064962,7.717643,9.050877,0.388906,0.496277,3.094451,0.992989,2.092888
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,4.000000
50%,0.000000,0.000000,0.000000,1.000000,27.000000,0.00000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,6.000000
75%,0.000000,1.000000,1.000000,1.000000,32.000000,1.00000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,2.000000,4.000000,0.000000,1.000000,10.000000,6.000000,8.000000
max,1.000000,1.000000,1.000000,1.000000,98.000000,1.00000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000


In [7]:
data['Diabetes_binary'].value_counts(normalize=True)

Diabetes_binary
0.0    0.847055
1.0    0.152945
Name: proportion, dtype: float64

In [8]:
X, y= data.drop('Diabetes_binary', axis=1).values, data['Diabetes_binary'].values
Xtrain, Xtest, ytrain, ytest=train_test_split(X,y, test_size=.1, random_state=42, shuffle=True)

In [27]:
model = Pipeline([('scalar', StandardScaler()), ('model', 'passthrough')])

params= [{'model': [RandomForestClassifier()],
          'model__max_depth':[ 7, 10, 15],
          'model__n_estimators': [50, 100, 150],
          'model__class_weight':[{0: 1, 1: v} for v in range(1, 5)],
          'model__random_state': [42]
         },
         
         {'model': [LogisticRegression()],
          'model__max_iter':[1000],
          'model__class_weight':[{0: 1, 1: v} for v in range(1, 5)],
          'model__random_state': [42]
         },
         {'model':[LinearSVC()],
          'model__C' : [0.1, 1, 10, 100],
          #'model__kernel':['rbf', 'linear'],
          'model__class_weight':[{0: 1, 1: v} for v in range(1, 5)],
          'model__random_state': [42]
         }
        ]

In [28]:
grid = GridSearchCV(model, params, cv =10,
                   scoring=make_scorer(f1_score),
                    n_jobs=-1,
                    refit=True,
                    verbose=10)
grid.fit(Xtrain, ytrain)



Fitting 10 folds for each of 56 candidates, totalling 560 fits


[CV 1/10; 1/56] START model=RandomForestClassifier(), model__class_weight={0: 1, 1: 1}, model__max_depth=7, model__n_estimators=50, model__random_state=42
[CV 2/10; 1/56] START model=RandomForestClassifier(), model__class_weight={0: 1, 1: 1}, model__max_depth=7, model__n_estimators=50, model__random_state=42
[CV 2/10; 1/56] END model=RandomForestClassifier(), model__class_weight={0: 1, 1: 1}, model__max_depth=7, model__n_estimators=50, model__random_state=42;, score=0.098 total time=   9.8s
[CV 1/10; 1/56] END model=RandomForestClassifier(), model__class_weight={0: 1, 1: 1}, model__max_depth=7, model__n_estimators=50, model__random_state=42;, score=0.094 total time=   9.8s
[CV 3/10; 1/56] START model=RandomForestClassifier(), model__class_weight={0: 1, 1: 1}, model__max_depth=7, model__n_estimators=50, model__random_state=42
[CV 4/10; 1/56] START model=RandomForestClassifier(), model__class_weight={0: 1, 1: 1}, model__max_depth=7, model__n_estimators=50, model__random_state=42
[CV 3/10

KeyboardInterrupt: 

In [29]:
ypred= grid.best_estimator_.predict(Xtest)
score= f1_score(ytest, ypred)  
print(f'F1 score: {score:.4f}')
print(f'accuracy score: {accuracy_score(ytest, ypred):.4f}')


AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'

In [26]:
grid.best_params_

{'model': LinearSVC(),
 'model__C': 1,
 'model__class_weight': {0: 1, 1: 4},
 'model__random_state': 42}

In [31]:

import sys

del large_dataframe
del raw_dataset

NameError: name 'large_dataframe' is not defined